# P01 - Imagen log-mel 128 x 512

Pipeline principal para CNN. En vez de colapsar el tiempo en estadisticas, conserva el espectrograma como matriz tiempo-frecuencia.

Este cambio explica el salto fuerte de private LB `0.37607` a `0.52257`: la CNN puede aprender patrones locales de audio que el baseline tabular no ve.


## Imports y configuracion


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import zipfile

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

ROOT = Path.cwd()
if ROOT.name == '02_preprocesamiento':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
RESULTS_DIR = ROOT / '02_preprocesamiento' / 'results'
INVESTIGATION = ROOT / 'investigation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(INVESTIGATION))

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 220)

from scripts.fat2019.data import CORRUPT_OR_BAD_LABEL_FILES, split_labels
from scripts.fat2019.features import read_wav_mono, log_mel_spectrogram, extract_log_mel_stats

from numpy.lib.format import read_magic, _read_array_header


def npz_array_headers(path: Path) -> pd.DataFrame:
    rows = []
    with zipfile.ZipFile(path) as zf:
        for info in zf.infolist():
            if not info.filename.endswith('.npy'):
                continue
            with zf.open(info) as f:
                version = read_magic(f)
                shape, fortran_order, dtype = _read_array_header(f, version)
            rows.append({
                'array': info.filename.replace('.npy', ''),
                'shape': str(shape),
                'dtype': str(dtype),
                'uncompressed_mb': round(info.file_size / 1024**2, 2),
                'compressed_mb': round(info.compress_size / 1024**2, 2),
            })
    return pd.DataFrame(rows)


## Inspeccion liviana del cache


In [2]:
cache_path = DATA_DIR / 'curated_logmel_image_m128_f512.npz'
test_cache_path = DATA_DIR / 'test_logmel_image_m128_f512.npz'

summary = pd.concat([
    npz_array_headers(cache_path).assign(cache='curated_logmel_image_m128_f512'),
    npz_array_headers(test_cache_path).assign(cache='test_logmel_image_m128_f512'),
], ignore_index=True)
display(summary[['cache', 'array', 'shape', 'dtype', 'compressed_mb']])
summary.to_csv(RESULTS_DIR / 'P01_logmel_image_512_summary.csv', index=False)


,cache,array,shape,dtype,compressed_mb
0,curated_logmel_image_m128_f512,x,"(4964, 128, 512)",float16,348.30
1,curated_logmel_image_m128_f512,fnames,"(4964,)",<U12,0.03
2,test_logmel_image_m128_f512,x,"(3361, 128, 512)",float16,291.01
3,test_logmel_image_m128_f512,fnames,"(3361,)",<U12,0.02


## Evidencia de impacto


In [3]:
evidence = pd.DataFrame([
    {'step': 'sklearn_logmel_c001', 'representation': 'stats log-mel', 'private_lb': 0.37607, 'reading': 'baseline tabular regularizado'},
    {'step': 'cnn_logmel_image', 'representation': 'imagen log-mel 128x512', 'private_lb': 0.52257, 'reading': 'preserva estructura tiempo-frecuencia'},
])
evidence['delta_vs_previous'] = evidence['private_lb'].diff()
display(evidence)


,step,representation,private_lb,reading,delta_vs_previous
0,sklearn_logmel_c001,stats log-mel,0.37607,baseline tabular regularizado,NaN
1,cnn_logmel_image,imagen log-mel 128x512,0.52257,preserva estructura tiempo-frecuencia,0.1465


## Decision

- Promover `128x512` como representacion base para CNN.
- Mantener caches separados para curated/test.
- No recalcular en notebooks de entrega salvo que cambien `n_mels`, `frames`, `hop_length` o normalizacion.
- El costo de cache se acepta porque habilita el salto de modelo tabular a CNN.
